# HCMC exact keyframes - Kaggle dual GPU

This notebook discovers videos beneath a **configurable Kaggle input**, runs
TransNetV2 once per worker, selects exact quality-aware shot and coverage
frames, and publishes resized JPEGs in one **flat** working output directory.

The final cell builds a resumable **progressive ZIP**. To remain within
Kaggle's working-storage limit it verifies archive bytes and CRCs, then
**deletes each verified batch** of source JPEGs. That deletion is intentional
and destructive; keep the final ZIP as the durable deliverable.

Package installation needs Kaggle **Internet** access. If Internet is disabled,
attach a compatible wheel bundle as a Kaggle Dataset and update the install
cell to use the **attached wheel** files.

With two visible CUDA devices the runner uses at most **two GPU** workers. It
falls back to one GPU and then a **CPU fallback**. Real two-GPU inference,
runtime, and the 15 GB input/storage envelope remain a **Kaggle acceptance**
test; local tests are model-free and cannot prove that runtime boundary.

`FAST_IO` defaults to `True` for Kaggle speed. It skips JPEG decode read-back
and per-JPEG file synchronization while retaining transaction journals, atomic
moves, and directory barriers. Set it to `False` for maximum crash safety.


In [1]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transnetv2-pytorch>=1.0.5,<2",
        "opencv-python-headless>=4.8,<6",
        "numpy>=1.26,<3",
    ],
    check=True,
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.7/32.7 MB 54.3 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transnetv2-pytorch>=1.0.5,<2', 'opencv-python-headless>=4.8,<6', 'numpy>=1.26,<3'], returncode=0)

In [2]:
import math
import os
import tempfile
from pathlib import Path

# Dataset version 1: https://www.kaggle.com/datasets/trieu241007/kis-skillpixel/versions/1
INPUT_PATH = Path("/kaggle/input/datasets/nguynminhphucaiwork/video-l29")
OUTPUT_DIR = Path("/kaggle/working/keyframes")
ARCHIVE_PATH = Path("/kaggle/working/keyframes.zip")
ARCHIVE_BATCH_SIZE = 128
SCENE_THRESHOLD = 0.5
COVERAGE_SECONDS = 1.0
CANDIDATE_WINDOW_SECONDS = 0.25
CANDIDATE_FPS = 5.0
IMAGE_SIZE = 736
JPEG_QUALITY = 92
MAX_WORKERS = 2
OVERWRITE = False
FAIL_FAST = False
FAST_IO = True


def format_elapsed(seconds):
    total_seconds = max(0, int(round(float(seconds))))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, remaining_seconds = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {remaining_seconds}s"
    if minutes:
        return f"{minutes}m {remaining_seconds}s"
    return f"{remaining_seconds}s"


def _require_finite_number(name, value, *, minimum=None, maximum=None):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{name} must be a finite number")
    number = float(value)
    if not math.isfinite(number):
        raise ValueError(f"{name} must be finite")
    if minimum is not None and number < minimum:
        raise ValueError(f"{name} must be at least {minimum}")
    if maximum is not None and number > maximum:
        raise ValueError(f"{name} must be at most {maximum}")


def _require_integer(name, value, *, minimum, maximum=None):
    if isinstance(value, bool) or not isinstance(value, int):
        raise ValueError(f"{name} must be an integer")
    if value < minimum or (maximum is not None and value > maximum):
        range_text = f"{minimum}..{maximum}" if maximum is not None else f">= {minimum}"
        raise ValueError(f"{name} must be {range_text}")


def _probe_writable_directory(directory, label):
    directory = Path(directory)
    try:
        directory.mkdir(parents=True, exist_ok=True)
        if not directory.is_dir():
            raise NotADirectoryError(directory)
        descriptor, probe_name = tempfile.mkstemp(
            prefix=".hcmc-keyframes-preflight-", dir=directory
        )
        os.close(descriptor)
        Path(probe_name).unlink()
    except OSError as error:
        raise RuntimeError(
            f"{label} is not usable for writes: {directory}: {error}"
        ) from error


def validate_notebook_configuration(
    *,
    input_path,
    output_dir,
    archive_path,
    archive_batch_size,
    scene_threshold,
    coverage_seconds,
    candidate_window_seconds,
    candidate_fps,
    image_size,
    jpeg_quality,
    max_workers,
    fast_io,
):
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    archive_path = Path(archive_path)
    if not input_path.exists():
        raise FileNotFoundError(f"INPUT_PATH does not exist: {input_path}")
    if not (input_path.is_file() or input_path.is_dir()):
        raise ValueError(f"INPUT_PATH must be a file or directory: {input_path}")
    if output_dir.exists() and not output_dir.is_dir():
        raise ValueError(f"OUTPUT_DIR must be a directory: {output_dir}")
    if archive_path.exists() and not archive_path.is_file():
        raise ValueError(f"ARCHIVE_PATH must be a file location: {archive_path}")

    input_resolved = input_path.resolve()
    output_resolved = output_dir.resolve()
    archive_resolved = archive_path.resolve()
    archive_state_resolved = {
        archive_path.with_name(f".{archive_path.name}.partial").resolve(),
        archive_path.with_name(f".{archive_path.name}.checkpoint").resolve(),
    }
    if output_resolved == archive_resolved:
        raise ValueError("OUTPUT_DIR and ARCHIVE_PATH must be different locations")
    if archive_resolved.is_relative_to(output_resolved):
        raise ValueError("ARCHIVE_PATH must not be inside OUTPUT_DIR")
    if output_resolved.is_relative_to(archive_resolved):
        raise ValueError("OUTPUT_DIR must not be inside ARCHIVE_PATH")
    if any(
        output_resolved == state_path
        or output_resolved.is_relative_to(state_path)
        for state_path in archive_state_resolved
    ):
        raise ValueError("OUTPUT_DIR conflicts with private archive recovery state")
    if input_resolved in {
        output_resolved,
        archive_resolved,
        *archive_state_resolved,
    }:
        raise ValueError(
            "INPUT_PATH must not equal OUTPUT_DIR, ARCHIVE_PATH, or archive recovery state"
        )
    if input_path.is_dir() and (
        output_resolved.is_relative_to(input_resolved)
        or archive_resolved.is_relative_to(input_resolved)
    ):
        raise ValueError("OUTPUT_DIR and ARCHIVE_PATH must not be inside INPUT_PATH")

    _require_finite_number(
        "SCENE_THRESHOLD", scene_threshold, minimum=0.0, maximum=1.0
    )
    _require_finite_number(
        "COVERAGE_SECONDS", coverage_seconds, minimum=0.000000001
    )
    _require_finite_number(
        "CANDIDATE_WINDOW_SECONDS",
        candidate_window_seconds,
        minimum=0.000000001,
    )
    _require_finite_number("CANDIDATE_FPS", candidate_fps, minimum=0.000000001)
    _require_integer("IMAGE_SIZE", image_size, minimum=1)
    _require_integer("JPEG_QUALITY", jpeg_quality, minimum=1, maximum=100)
    _require_integer("MAX_WORKERS", max_workers, minimum=1, maximum=2)
    _require_integer("ARCHIVE_BATCH_SIZE", archive_batch_size, minimum=1)
    if not isinstance(fast_io, bool):
        raise ValueError("FAST_IO must be a boolean")

    _probe_writable_directory(
        output_dir if output_dir.exists() else output_dir.parent,
        "OUTPUT_DIR parent",
    )
    _probe_writable_directory(archive_path.parent, "ARCHIVE_PATH parent")


In [3]:
import shutil
import subprocess
import torch

validate_notebook_configuration(
    input_path=INPUT_PATH,
    output_dir=OUTPUT_DIR,
    archive_path=ARCHIVE_PATH,
    archive_batch_size=ARCHIVE_BATCH_SIZE,
    scene_threshold=SCENE_THRESHOLD,
    coverage_seconds=COVERAGE_SECONDS,
    candidate_window_seconds=CANDIDATE_WINDOW_SECONDS,
    candidate_fps=CANDIDATE_FPS,
    image_size=IMAGE_SIZE,
    jpeg_quality=JPEG_QUALITY,
    max_workers=MAX_WORKERS,
    fast_io=FAST_IO,
)

for command in ("ffmpeg", "ffprobe"):
    path = shutil.which(command)
    if path is None:
        raise RuntimeError(f"{command} is required but was not found")
    subprocess.run([path, "-version"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"cuda:{index}:", torch.cuda.get_device_name(index))
print("FAST_IO:", FAST_IO)


PyTorch: 2.10.0+cu128
CUDA available: True
CUDA devices: 2
cuda:0: Tesla T4
cuda:1: Tesla T4
FAST_IO: True


In [4]:
RUNNER_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib\nimport json\nimport math\nimport multiprocessing\nimport os\nimport queue\nimport re\nimport subprocess\nimport time\nfrom collections import Counter\nfrom dataclasses import asdict, dataclass\nfrom fractions import Fraction\nfrom pathlib import Path\nfrom typing import Mapping, Sequence\n\nimport cv2\nimport numpy as np\n\n\nVIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}\ntorch = None\n\n\n@dataclass(frozen=True)\nclass RunnerConfig:\n    input_path: Path = Path("/kaggle/input/kis-skillpixel")\n    output_dir: Path = Path("/kaggle/working/keyframes")\n    scene_threshold: float = 0.5\n    coverage_seconds: float = 1.0\n    candidate_window_seconds: float = 0.25\n    candidate_fps: float = 5.0\n    image_size: int = 736\n    jpeg_quality: int = 92\n    max_workers: int = 2\n    overwrite: bool = False\n    fail_fast: bool = False\n    fast_io: bool = False\n    sharpness_weight: float = 0.60\n    exposure_weight: float = 0.25\n    contrast_weight: float = 0.15\n\n    def __post_init__(self) -> None:\n        if not 0.0 <= self.scene_threshold <= 1.0:\n            raise ValueError("scene_threshold must be between 0 and 1")\n        if min(self.coverage_seconds, self.candidate_window_seconds, self.candidate_fps) <= 0:\n            raise ValueError("selection timing values must be positive")\n        if self.image_size < 1 or not 1 <= self.jpeg_quality <= 100:\n            raise ValueError("image_size and jpeg_quality are invalid")\n        if not 1 <= self.max_workers <= 2:\n            raise ValueError("max_workers must be 1 or 2")\n        if not isinstance(self.fast_io, bool):\n            raise ValueError("fast_io must be a boolean")\n        weights = (self.sharpness_weight, self.exposure_weight, self.contrast_weight)\n        if any(weight < 0 for weight in weights) or abs(sum(weights) - 1.0) > 1e-9:\n            raise ValueError("quality weights must be non-negative and sum to 1")\n\n    @classmethod\n    def from_mapping(cls, values: Mapping[str, object]) -> "RunnerConfig":\n        converted = dict(values)\n        converted["input_path"] = Path(str(converted["input_path"]))\n        converted["output_dir"] = Path(str(converted["output_dir"]))\n        return cls(**converted)\n\n\n@dataclass(frozen=True)\nclass VideoInfo:\n    video_id: str\n    path: Path\n    width: int\n    height: int\n    fps: float\n    frame_count: int\n    duration: float\n    size_bytes: int\n\n\n@dataclass(frozen=True)\nclass Shot:\n    video_id: str\n    shot_id: int\n    start_frame: int\n    end_frame: int\n    fps: float\n\n    @property\n    def midpoint(self) -> int:\n        return (self.start_frame + self.end_frame) // 2\n\n    @property\n    def duration(self) -> float:\n        return (self.end_frame - self.start_frame + 1) / self.fps\n\n\n@dataclass(frozen=True)\nclass SelectionTarget:\n    shot_id: int\n    target_frame: int\n    reasons: tuple[str, ...]\n\n\n@dataclass(frozen=True)\nclass FrameQuality:\n    sharpness: float\n    exposure: float\n    contrast: float\n\n\n@dataclass(frozen=True)\nclass FrameCandidate:\n    frame_idx: int\n    target_frame: int\n    shot_id: int\n    quality: FrameQuality\n\n    def filename(self, video_id: str) -> str:\n        return f"{video_id}_s{self.shot_id:06d}_f{self.frame_idx:09d}.jpg"\n\n\n@dataclass(frozen=True)\nclass VideoResult:\n    video_id: str\n    status: str\n    keyframe_count: int\n    device: str\n    error: str\n\n\ndef discover_videos(input_path: Path) -> list[Path]:\n    input_path = Path(input_path)\n    if input_path.is_file():\n        if input_path.suffix.lower() not in VIDEO_EXTENSIONS:\n            raise FileNotFoundError(f"No supported videos found at {input_path}")\n        return [input_path]\n    if not input_path.is_dir():\n        raise FileNotFoundError(f"No supported videos found at {input_path}")\n    videos = [path for path in input_path.rglob("*") if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS]\n    videos.sort(\n        key=lambda path: (\n            path.relative_to(input_path).as_posix().casefold(),\n            path.relative_to(input_path).as_posix(),\n        )\n    )\n    if not videos:\n        raise FileNotFoundError(f"No supported videos found at {input_path}")\n    return videos\n\n\ndef _sanitize_stem(stem: str) -> str:\n    sanitized = re.sub(r"[^a-z0-9]+", "-", stem.casefold()).strip("-")\n    return sanitized or "video"\n\n\ndef assign_video_ids(paths: Sequence[Path], root: Path) -> dict[Path, str]:\n    root = Path(root)\n    stems = {path: _sanitize_stem(path.stem) for path in paths}\n    counts = Counter(stems.values())\n    result: dict[Path, str] = {}\n    for path in paths:\n        stem = stems[path]\n        if counts[stem] == 1:\n            result[path] = stem\n            continue\n        try:\n            relative = path.relative_to(root).as_posix()\n        except ValueError:\n            relative = path.resolve().as_posix()\n        suffix = hashlib.sha256(relative.encode("utf-8")).hexdigest()[:8]\n        result[path] = f"{stem}-{suffix}"\n    return result\n\n\ndef probe_video(path: Path, video_id: str) -> VideoInfo:\n    path = Path(path)\n    command = [\n        "ffprobe",\n        "-v", "error",\n        "-select_streams", "v:0",\n        "-show_entries",\n        "stream=width,height,avg_frame_rate,nb_frames,duration:format=duration",\n        "-of", "json",\n        str(path),\n    ]\n    try:\n        completed = subprocess.run(command, check=True, capture_output=True, text=True)\n        payload = json.loads(completed.stdout)\n        streams = payload.get("streams", [])\n        if not streams:\n            raise ValueError("no video stream")\n        stream = streams[0]\n        width = int(stream["width"])\n        height = int(stream["height"])\n        fps = float(Fraction(str(stream["avg_frame_rate"])))\n        frame_count = int(stream["nb_frames"])\n        duration_value = stream.get("duration")\n        if duration_value in (None, "N/A"):\n            duration_value = payload.get("format", {}).get("duration")\n        duration = float(duration_value)\n        size_bytes = path.stat().st_size\n        values = (width, height, fps, frame_count, duration, size_bytes)\n        if any(not math.isfinite(float(value)) or value <= 0 for value in values):\n            raise ValueError("width, height, fps, frame count, duration, and size must be positive")\n    except Exception as error:\n        raise ValueError(f"Could not probe video {path}: {error}") from error\n    return VideoInfo(video_id, path, width, height, fps, frame_count, duration, size_bytes)\n\n\ndef load_transnet(device: str) -> object:\n    module = importlib.import_module("transnetv2_pytorch")\n    return module.TransNetV2(device=device)\n\n\ndef normalize_scenes(raw: object, video: VideoInfo) -> list[Shot]:\n    if raw is None:\n        raise ValueError("scene detector returned None")\n    entries = raw.get("scenes", []) if isinstance(raw, Mapping) else raw\n    shots: list[Shot] = []\n    for entry in entries:\n        if isinstance(entry, Mapping):\n            start, end = entry["start_frame"], entry["end_frame"]\n        else:\n            start, end = entry\n        start_frame = max(0, int(start))\n        end_frame = min(video.frame_count - 1, int(end))\n        if end_frame < start_frame:\n            continue\n        shots.append(Shot(video.video_id, len(shots) + 1, start_frame, end_frame, video.fps))\n    if not shots and video.frame_count > 0:\n        return [Shot(video.video_id, 1, 0, video.frame_count - 1, video.fps)]\n    return shots\n\n\ndef plan_devices(cuda_available: bool, cuda_count: int, max_workers: int) -> tuple[str, ...]:\n    if max_workers < 1:\n        raise ValueError("max_workers must be positive")\n    if not cuda_available or cuda_count < 1:\n        return ("cpu",)\n    count = min(2, max_workers, cuda_count)\n    return tuple(f"cuda:{index}" for index in range(count))\n\n\ndef shard_videos(videos: Sequence[VideoInfo], worker_count: int) -> tuple[tuple[VideoInfo, ...], ...]:\n    if worker_count < 1:\n        raise ValueError("worker_count must be positive")\n    shards: list[list[VideoInfo]] = [[] for _ in range(worker_count)]\n    totals = [0] * worker_count\n    for video in sorted(\n        videos,\n        key=lambda item: (\n            -(item.frame_count * item.width * item.height),\n            item.path.as_posix().casefold(),\n            item.path.as_posix(),\n        ),\n    ):\n        worker = min(range(worker_count), key=lambda index: (totals[index], index))\n        shards[worker].append(video)\n        totals[worker] += video.frame_count * video.width * video.height\n    return tuple(\n        tuple(\n            sorted(\n                shard,\n                key=lambda item: (\n                    item.path.as_posix().casefold(), item.path.as_posix()\n                ),\n            )\n        )\n        for shard in shards\n    )\n\n\ndef generate_targets(shot: Shot, config: RunnerConfig) -> tuple[SelectionTarget, ...]:\n    reasons: dict[int, list[str]] = {}\n\n    def add(frame_idx: int, reason: str) -> None:\n        bucket = reasons.setdefault(frame_idx, [])\n        if reason not in bucket:\n            bucket.append(reason)\n\n    add(shot.start_frame, "shot_start")\n    add(shot.midpoint, "representative")\n    add(shot.end_frame, "shot_end")\n    if shot.duration > config.coverage_seconds:\n        step = max(1, round(config.coverage_seconds * shot.fps))\n        for frame_idx in range(shot.start_frame, shot.end_frame + 1, step):\n            add(frame_idx, "coverage")\n    return tuple(\n        SelectionTarget(shot.shot_id, frame_idx, tuple(reasons[frame_idx]))\n        for frame_idx in sorted(reasons)\n    )\n\n\ndef candidate_indices(\n    target: SelectionTarget, shot: Shot, config: RunnerConfig\n) -> tuple[int, ...]:\n    half_width = max(0, round(config.candidate_window_seconds * shot.fps))\n    start = max(shot.start_frame, target.target_frame - half_width)\n    end = min(shot.end_frame, target.target_frame + half_width)\n    stride = max(1, round(shot.fps / config.candidate_fps))\n    indices = set(range(start, end + 1, stride))\n    indices.update((start, target.target_frame, end))\n    return tuple(sorted(indices))\n\n\ndef measure_quality(frame: np.ndarray) -> FrameQuality:\n    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())\n    exposure = max(0.0, 1.0 - abs(float(gray.mean()) - 127.5) / 127.5)\n    contrast = min(1.0, float(gray.std()) / 64.0)\n    return FrameQuality(sharpness, exposure, contrast)\n\n\ndef choose_best_candidate(\n    candidates: Sequence[FrameCandidate], config: RunnerConfig\n) -> FrameCandidate:\n    if not candidates:\n        raise ValueError("at least one candidate is required")\n    logged = [math.log1p(max(0.0, item.quality.sharpness)) for item in candidates]\n    low, high = min(logged), max(logged)\n    normalized = [1.0] * len(logged) if high == low else [\n        (value - low) / (high - low) for value in logged\n    ]\n    scored = []\n    for candidate, sharpness in zip(candidates, normalized, strict=True):\n        score = (\n            config.sharpness_weight * sharpness\n            + config.exposure_weight * candidate.quality.exposure\n            + config.contrast_weight * candidate.quality.contrast\n        )\n        scored.append(\n            (score, -abs(candidate.frame_idx - candidate.target_frame), -candidate.frame_idx, candidate)\n        )\n    return max(scored, key=lambda item: item[:3])[3]\n\n\ndef resized_dimensions(width: int, height: int, image_size: int) -> tuple[int, int]:\n    if width < 1 or height < 1 or image_size < 1:\n        raise ValueError("image dimensions and image_size must be positive")\n    scale = image_size / max(width, height)\n    return max(1, round(width * scale)), max(1, round(height * scale))\n\n\ndef resize_long_side(frame: np.ndarray, image_size: int) -> np.ndarray:\n    height, width = frame.shape[:2]\n    target_width, target_height = resized_dimensions(width, height, image_size)\n    interpolation = cv2.INTER_AREA if target_width < width or target_height < height else cv2.INTER_LINEAR\n    return cv2.resize(frame, (target_width, target_height), interpolation=interpolation)\n\n\ndef _canonical_keyframe_pattern(video_id: str) -> re.Pattern[str]:\n    return re.compile(rf"^{re.escape(video_id)}_s\\d{{6}}_f\\d{{9}}\\.jpg$")\n\n\ndef keyframes_for_video(output_dir: Path, video_id: str) -> tuple[Path, ...]:\n    if not output_dir.exists():\n        return ()\n    pattern = _canonical_keyframe_pattern(video_id)\n    return tuple(\n        sorted(\n            (path for path in output_dir.iterdir() if path.is_file() and pattern.fullmatch(path.name)),\n            key=lambda path: path.name,\n        )\n    )\n\n\nTRANSACTION_DIRECTORY_NAME = ".transactions"\nTRANSACTION_JOURNAL_NAME = "journal.json"\nTRANSACTION_VERSION = 1\n\n\ndef _fsync_file(path: Path) -> None:\n    with Path(path).open("r+b") as stream:\n        os.fsync(stream.fileno())\n\n\ndef _fsync_directory(path: Path) -> None:\n    if os.name != "posix":\n        return\n    flags = os.O_RDONLY | getattr(os, "O_DIRECTORY", 0)\n    descriptor = os.open(os.fspath(Path(path)), flags)\n    try:\n        os.fsync(descriptor)\n    finally:\n        os.close(descriptor)\n\n\ndef _mkdir_durable(path: Path) -> None:\n    path = Path(path)\n    if path.exists():\n        if not path.is_dir():\n            raise NotADirectoryError(f"Transaction path is not a directory: {path}")\n        return\n    path.mkdir()\n    _fsync_directory(path.parent)\n\n\ndef _transaction_paths(output_dir: Path, video_id: str) -> tuple[Path, Path, Path, Path]:\n    if not re.fullmatch(r"[a-z0-9][a-z0-9-]*", video_id):\n        raise ValueError(f"Unsafe video_id for transaction state: {video_id!r}")\n    transaction_dir = Path(output_dir) / TRANSACTION_DIRECTORY_NAME / video_id\n    return (\n        transaction_dir,\n        transaction_dir / "staging",\n        transaction_dir / "backup",\n        transaction_dir / TRANSACTION_JOURNAL_NAME,\n    )\n\n\ndef _sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _manifest_entry(path: Path) -> dict[str, object]:\n    path = Path(path)\n    return {\n        "name": path.name,\n        "size": path.stat().st_size,\n        "sha256": _sha256_file(path),\n    }\n\n\ndef _manifest_for_paths(paths: Sequence[Path]) -> list[dict[str, object]]:\n    return [_manifest_entry(path) for path in sorted(paths, key=lambda item: item.name)]\n\n\ndef _entry_matches(path: Path, entry: Mapping[str, object]) -> bool:\n    return (\n        path.is_file()\n        and path.stat().st_size == entry["size"]\n        and _sha256_file(path) == entry["sha256"]\n    )\n\n\ndef _write_transaction_journal(journal_path: Path, payload: Mapping[str, object]) -> None:\n    temporary = journal_path.with_name(f".{journal_path.name}.tmp")\n    with temporary.open("w", encoding="utf-8", newline="\\n") as stream:\n        json.dump(dict(payload), stream, sort_keys=True, separators=(",", ":"))\n        stream.write("\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temporary, journal_path)\n    _fsync_directory(journal_path.parent)\n\n\ndef _validate_manifest_entries(\n    entries: object, video_id: str, *, label: str\n) -> list[dict[str, object]]:\n    if not isinstance(entries, list):\n        raise ValueError(f"{label} must be a list")\n    pattern = _canonical_keyframe_pattern(video_id)\n    validated: list[dict[str, object]] = []\n    for raw in entries:\n        if not isinstance(raw, dict) or set(raw) != {"name", "size", "sha256"}:\n            raise ValueError(f"{label} contains an invalid manifest entry")\n        name, size, digest = raw["name"], raw["size"], raw["sha256"]\n        if not isinstance(name, str) or pattern.fullmatch(name) is None:\n            raise ValueError(f"{label} contains an unsafe canonical name")\n        if isinstance(size, bool) or not isinstance(size, int) or size < 0:\n            raise ValueError(f"{label} contains an invalid byte size")\n        if not isinstance(digest, str) or re.fullmatch(r"[0-9a-f]{64}", digest) is None:\n            raise ValueError(f"{label} contains an invalid checksum")\n        validated.append({"name": name, "size": size, "sha256": digest})\n    names = [entry["name"] for entry in validated]\n    if names != sorted(names) or len(names) != len(set(names)):\n        raise ValueError(f"{label} names must be unique and sorted")\n    return validated\n\n\ndef _read_transaction_journal(journal_path: Path, video_id: str) -> dict[str, object]:\n    try:\n        payload = json.loads(journal_path.read_text(encoding="utf-8"))\n        if not isinstance(payload, dict) or set(payload) != {\n            "version", "video_id", "phase", "old", "new_names", "manifest"\n        }:\n            raise ValueError("journal fields are invalid")\n        if payload["version"] != TRANSACTION_VERSION or payload["video_id"] != video_id:\n            raise ValueError("journal identity is invalid")\n        if payload["phase"] not in {"prepared", "old_backed_up", "committed"}:\n            raise ValueError("journal phase is invalid")\n        old_entries = _validate_manifest_entries(payload["old"], video_id, label="old")\n        manifest = _validate_manifest_entries(\n            payload["manifest"], video_id, label="manifest"\n        )\n        new_names = payload["new_names"]\n        pattern = _canonical_keyframe_pattern(video_id)\n        if (\n            not isinstance(new_names, list)\n            or any(not isinstance(name, str) or pattern.fullmatch(name) is None for name in new_names)\n            or new_names != sorted(new_names)\n            or len(new_names) != len(set(new_names))\n        ):\n            raise ValueError("new_names must be unique sorted canonical names")\n        if payload["phase"] in {"old_backed_up", "committed"}:\n            if [entry["name"] for entry in manifest] != new_names:\n                raise ValueError("durable phase requires a complete new manifest")\n        elif manifest and [entry["name"] for entry in manifest] != new_names:\n            raise ValueError("prepared manifest does not match new_names")\n        payload["old"] = old_entries\n        payload["manifest"] = manifest\n        return payload\n    except Exception as error:\n        raise RuntimeError(\n            f"Cannot read transaction journal for {video_id}: {error}"\n        ) from error\n\n\ndef _validate_manifest_files(\n    directory: Path,\n    video_id: str,\n    entries: Sequence[Mapping[str, object]],\n    *,\n    label: str,\n) -> tuple[Path, ...]:\n    expected_names = [str(entry["name"]) for entry in entries]\n    actual = keyframes_for_video(directory, video_id)\n    if [path.name for path in actual] != expected_names:\n        raise RuntimeError(\n            f"{label} for {video_id} has canonical names that do not match the manifest"\n        )\n    for path, entry in zip(actual, entries, strict=True):\n        if not _entry_matches(path, entry):\n            raise RuntimeError(\n                f"{label} for {video_id} failed committed manifest validation: {path.name}"\n            )\n    return actual\n\n\ndef _validate_known_transaction_contents(\n    directory: Path, allowed_names: set[str], *, label: str\n) -> None:\n    if not directory.exists():\n        return\n    if not directory.is_dir():\n        raise RuntimeError(f"Transaction {label} is not a directory: {directory}")\n    unexpected = sorted(path.name for path in directory.iterdir() if path.name not in allowed_names)\n    if unexpected:\n        raise RuntimeError(\n            f"Transaction {label} contains unexpected state: {\', \'.join(unexpected)}"\n        )\n\n\ndef _remove_rolled_back_transaction(\n    output_dir: Path, video_id: str, journal: Mapping[str, object]\n) -> None:\n    transaction_dir, stage_dir, backup_dir, journal_path = _transaction_paths(\n        output_dir, video_id\n    )\n    new_names = set(journal["new_names"])\n    old_names = {str(entry["name"]) for entry in journal["old"]}\n    _validate_known_transaction_contents(stage_dir, new_names, label="staging")\n    _validate_known_transaction_contents(backup_dir, old_names, label="backup")\n    for name in sorted(new_names):\n        path = stage_dir / name\n        if path.exists():\n            path.unlink()\n    for name in sorted(old_names):\n        path = backup_dir / name\n        if path.exists():\n            raise RuntimeError(f"Rollback left a backup behind for {video_id}: {name}")\n    if stage_dir.exists():\n        stage_dir.rmdir()\n    if backup_dir.exists():\n        backup_dir.rmdir()\n    temporary = journal_path.with_name(f".{journal_path.name}.tmp")\n    if temporary.exists():\n        temporary.unlink()\n    journal_path.unlink()\n    _fsync_directory(transaction_dir)\n    transaction_dir.rmdir()\n    transaction_root = transaction_dir.parent\n    _fsync_directory(transaction_root)\n    try:\n        transaction_root.rmdir()\n    except OSError:\n        pass\n    _fsync_directory(output_dir)\n\n\ndef _rollback_video_transaction(\n    output_dir: Path, video_id: str, journal: Mapping[str, object]\n) -> None:\n    transaction_dir, stage_dir, backup_dir, _ = _transaction_paths(output_dir, video_id)\n    del transaction_dir, stage_dir\n    old_entries = {str(entry["name"]): entry for entry in journal["old"]}\n    new_entries = {str(entry["name"]): entry for entry in journal["manifest"]}\n    phase = str(journal["phase"])\n    try:\n        if phase == "old_backed_up":\n            for name in journal["new_names"]:\n                destination = Path(output_dir) / name\n                if not destination.exists():\n                    continue\n                old_entry = old_entries.get(name)\n                backup = backup_dir / name\n                if old_entry is not None and not backup.exists() and _entry_matches(destination, old_entry):\n                    continue\n                new_entry = new_entries.get(name)\n                if new_entry is None or not _entry_matches(destination, new_entry):\n                    raise RuntimeError(\n                        f"Published file changed during rollback for {video_id}: {name}"\n                    )\n                destination.unlink()\n\n        for name, entry in old_entries.items():\n            backup = backup_dir / name\n            destination = Path(output_dir) / name\n            if backup.exists():\n                if not _entry_matches(backup, entry):\n                    raise RuntimeError(f"Backup checksum mismatch for {video_id}: {name}")\n                backup.replace(destination)\n                _fsync_file(destination)\n            elif not _entry_matches(destination, entry):\n                raise RuntimeError(f"Previous file cannot be restored for {video_id}: {name}")\n        _fsync_directory(Path(output_dir))\n        _fsync_directory(backup_dir)\n        _validate_manifest_files(\n            Path(output_dir), video_id, list(old_entries.values()), label="rolled-back set"\n        )\n        _remove_rolled_back_transaction(Path(output_dir), video_id, journal)\n    except BaseException as error:\n        if isinstance(error, (KeyboardInterrupt, SystemExit)):\n            raise\n        raise RuntimeError(\n            f"Could not finish rollback for {video_id}; preserved transaction state: {error}"\n        ) from error\n\n\ndef _cleanup_committed_working_state(\n    output_dir: Path, video_id: str, journal: Mapping[str, object]\n) -> None:\n    _, stage_dir, backup_dir, _ = _transaction_paths(output_dir, video_id)\n    new_names = set(journal["new_names"])\n    old_names = {str(entry["name"]) for entry in journal["old"]}\n    _validate_known_transaction_contents(stage_dir, new_names, label="staging")\n    _validate_known_transaction_contents(backup_dir, old_names, label="backup")\n    for name in sorted(new_names):\n        staged = stage_dir / name\n        if staged.exists():\n            staged.unlink()\n    for name in sorted(old_names):\n        saved = backup_dir / name\n        if saved.exists():\n            saved.unlink()\n    if stage_dir.exists():\n        stage_dir.rmdir()\n    if backup_dir.exists():\n        backup_dir.rmdir()\n    _fsync_directory(stage_dir.parent)\n\n\ndef recover_video_transaction(\n    output_dir: Path, video_id: str, *, allow_legacy: bool = False\n) -> tuple[Path, ...] | None:\n    output_dir = Path(output_dir)\n    transaction_dir, stage_dir, backup_dir, journal_path = _transaction_paths(\n        output_dir, video_id\n    )\n    if not transaction_dir.exists():\n        existing = keyframes_for_video(output_dir, video_id)\n        if existing and not allow_legacy:\n            raise RuntimeError(\n                f"Canonical keyframes for {video_id} exist without a valid committed transaction; "\n                "rerun with overwrite only if rebuilding this video is intended"\n            )\n        return None\n    if not journal_path.is_file():\n        allowed = {stage_dir.name, backup_dir.name}\n        unexpected = [path for path in transaction_dir.iterdir() if path.name not in allowed]\n        safe_empty = (\n            not unexpected\n            and all(not path.exists() or (path.is_dir() and not any(path.iterdir())) for path in (stage_dir, backup_dir))\n        )\n        if safe_empty:\n            for path in (stage_dir, backup_dir):\n                if path.exists():\n                    path.rmdir()\n            transaction_dir.rmdir()\n            try:\n                transaction_dir.parent.rmdir()\n            except OSError:\n                pass\n            return recover_video_transaction(output_dir, video_id, allow_legacy=allow_legacy)\n        raise RuntimeError(\n            f"Transaction state for {video_id} has no durable journal; preserved {transaction_dir}"\n        )\n    journal = _read_transaction_journal(journal_path, video_id)\n    if journal["phase"] == "committed":\n        paths = _validate_manifest_files(\n            output_dir, video_id, journal["manifest"], label="committed manifest"\n        )\n        try:\n            _cleanup_committed_working_state(output_dir, video_id, journal)\n        except Exception:\n            pass\n        return paths\n    _rollback_video_transaction(output_dir, video_id, journal)\n    return None\n\n\ndef begin_video_transaction(\n    output_dir: Path,\n    video_id: str,\n    new_names: Sequence[str],\n    *,\n    overwrite: bool,\n) -> Path:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    _fsync_directory(output_dir.parent)\n    existing_commit = recover_video_transaction(\n        output_dir, video_id, allow_legacy=overwrite\n    )\n    if existing_commit is not None and not overwrite:\n        raise FileExistsError(f"Committed keyframes already exist for {video_id}")\n    existing = keyframes_for_video(output_dir, video_id)\n    if existing and not overwrite:\n        raise FileExistsError(f"Keyframes already exist for {video_id}")\n    pattern = _canonical_keyframe_pattern(video_id)\n    names = sorted(new_names)\n    if (\n        not names\n        or len(names) != len(set(names))\n        or any(pattern.fullmatch(name) is None for name in names)\n    ):\n        raise ValueError(f"Transaction for {video_id} requires unique canonical new names")\n    transaction_dir, stage_dir, backup_dir, journal_path = _transaction_paths(\n        output_dir, video_id\n    )\n    transaction_root = transaction_dir.parent\n    _mkdir_durable(transaction_root)\n    _mkdir_durable(transaction_dir)\n    _mkdir_durable(stage_dir)\n    _mkdir_durable(backup_dir)\n    journal = {\n        "version": TRANSACTION_VERSION,\n        "video_id": video_id,\n        "phase": "prepared",\n        "old": _manifest_for_paths(existing),\n        "new_names": names,\n        "manifest": [],\n    }\n    _write_transaction_journal(journal_path, journal)\n    return stage_dir\n\n\ndef publish_staged_video(\n    stage_dir: Path,\n    output_dir: Path,\n    video_id: str,\n    overwrite: bool,\n    *,\n    fast_io: bool = False,\n) -> tuple[Path, ...]:\n    output_dir = Path(output_dir)\n    stage_dir = Path(stage_dir)\n    transaction_dir, internal_stage, backup_dir, journal_path = _transaction_paths(\n        output_dir, video_id\n    )\n    if stage_dir != internal_stage:\n        source_paths = keyframes_for_video(stage_dir, video_id)\n        internal_stage = begin_video_transaction(\n            output_dir,\n            video_id,\n            tuple(path.name for path in source_paths),\n            overwrite=overwrite,\n        )\n        for source in source_paths:\n            source.replace(internal_stage / source.name)\n        _fsync_directory(internal_stage)\n        stage_dir = internal_stage\n    if not journal_path.is_file():\n        raise RuntimeError(f"Prepared transaction journal is missing for {video_id}")\n    journal = _read_transaction_journal(journal_path, video_id)\n    if journal["phase"] != "prepared":\n        raise RuntimeError(\n            f"Transaction for {video_id} is in phase {journal[\'phase\']!r}, not prepared"\n        )\n    staged_files = tuple(\n        sorted((path for path in stage_dir.iterdir() if path.is_file()), key=lambda path: path.name)\n    )\n    if [path.name for path in staged_files] != journal["new_names"]:\n        raise RuntimeError(f"Staged files do not match prepared transaction for {video_id}")\n    journal["manifest"] = _manifest_for_paths(staged_files)\n    _write_transaction_journal(journal_path, journal)\n\n    old_entries = {str(entry["name"]): entry for entry in journal["old"]}\n    for name, entry in old_entries.items():\n        original = output_dir / name\n        if not _entry_matches(original, entry):\n            raise RuntimeError(f"Prior canonical file changed before backup for {video_id}: {name}")\n        saved = backup_dir / name\n        original.replace(saved)\n        if not fast_io:\n            _fsync_file(saved)\n    _fsync_directory(output_dir)\n    _fsync_directory(backup_dir)\n    journal["phase"] = "old_backed_up"\n    _write_transaction_journal(journal_path, journal)\n\n    published: list[Path] = []\n    for source in staged_files:\n        destination = output_dir / source.name\n        source.replace(destination)\n        if not fast_io:\n            _fsync_file(destination)\n        published.append(destination)\n    _fsync_directory(output_dir)\n    _fsync_directory(stage_dir)\n    _validate_manifest_files(\n        output_dir,\n        video_id,\n        journal["manifest"],\n        label="published set",\n    )\n    journal["phase"] = "committed"\n    _write_transaction_journal(journal_path, journal)\n    try:\n        _cleanup_committed_working_state(output_dir, video_id, journal)\n    except Exception:\n        pass\n    return tuple(published)\n\n\ndef _decode_candidate_qualities(\n    video: VideoInfo, required_indices: set[int]\n) -> dict[int, FrameQuality]:\n    capture = cv2.VideoCapture(str(video.path))\n    if not capture.isOpened():\n        capture.release()\n        raise ValueError(f"Could not open video: {video.path}")\n    qualities: dict[int, FrameQuality] = {}\n    source_index = 0\n    try:\n        while True:\n            success, frame = capture.read()\n            if not success:\n                break\n            if source_index in required_indices:\n                qualities[source_index] = measure_quality(frame)\n            source_index += 1\n    finally:\n        capture.release()\n    return qualities\n\n\ndef select_winners(\n    plans: Sequence[tuple[Shot, SelectionTarget, tuple[int, ...]]],\n    qualities: Mapping[int, FrameQuality],\n    config: RunnerConfig,\n) -> tuple[FrameCandidate, ...]:\n    selected: dict[tuple[int, int], FrameCandidate] = {}\n    for shot, target, indices in plans:\n        candidates = [\n            FrameCandidate(\n                frame_idx,\n                target.target_frame,\n                shot.shot_id,\n                qualities[frame_idx],\n            )\n            for frame_idx in indices\n            if frame_idx in qualities\n        ]\n        if not candidates:\n            raise ValueError(\n                f"No decoded candidates for {shot.video_id} shot {shot.shot_id} "\n                f"target {target.target_frame}"\n            )\n        winner = choose_best_candidate(candidates, config)\n        selected.setdefault((winner.shot_id, winner.frame_idx), winner)\n    return tuple(\n        sorted(selected.values(), key=lambda item: (item.shot_id, item.frame_idx))\n    )\n\n\ndef _write_winning_frames(\n    video: VideoInfo,\n    selected: Sequence[FrameCandidate],\n    stage_dir: Path,\n    config: RunnerConfig,\n) -> None:\n    by_index: dict[int, list[FrameCandidate]] = {}\n    for candidate in selected:\n        by_index.setdefault(candidate.frame_idx, []).append(candidate)\n    capture = cv2.VideoCapture(str(video.path))\n    if not capture.isOpened():\n        capture.release()\n        raise ValueError(f"Could not open video: {video.path}")\n    remaining = {\n        candidate.filename(video.video_id)\n        for candidate in selected\n    }\n    expected_width, expected_height = resized_dimensions(\n        video.width, video.height, config.image_size\n    )\n    source_index = 0\n    try:\n        while remaining:\n            success, frame = capture.read()\n            if not success:\n                break\n            for candidate in by_index.get(source_index, ()):\n                destination = stage_dir / candidate.filename(video.video_id)\n                resized = resize_long_side(frame, config.image_size)\n                if resized.shape[:2] != (expected_height, expected_width):\n                    raise ValueError(f"Invalid resized keyframe: {destination}")\n                if not cv2.imwrite(\n                    str(destination),\n                    resized,\n                    [cv2.IMWRITE_JPEG_QUALITY, config.jpeg_quality],\n                ):\n                    raise ValueError(f"Could not write keyframe: {destination}")\n                if not config.fast_io:\n                    validated = cv2.imread(str(destination))\n                    if (\n                        validated is None\n                        or validated.shape[:2] != (expected_height, expected_width)\n                    ):\n                        raise ValueError(f"Invalid staged keyframe: {destination}")\n                remaining.remove(destination.name)\n            source_index += 1\n    finally:\n        capture.release()\n    if remaining:\n        raise ValueError(\n            f"Could not decode winning frames for {video.video_id}: "\n            f"{\', \'.join(sorted(remaining))}"\n        )\n\n\ndef extract_selected_frames(\n    video: VideoInfo, shots: Sequence[Shot], config: RunnerConfig\n) -> tuple[Path, ...]:\n    plans = [\n        (shot, target, candidate_indices(target, shot, config))\n        for shot in shots\n        for target in generate_targets(shot, config)\n    ]\n    required_indices = {frame_idx for _, _, indices in plans for frame_idx in indices}\n    qualities = _decode_candidate_qualities(video, required_indices)\n    selected = select_winners(plans, qualities, config)\n    if not selected:\n        raise ValueError(f"No keyframes selected for {video.video_id}")\n\n    stage_dir = begin_video_transaction(\n        config.output_dir,\n        video.video_id,\n        tuple(candidate.filename(video.video_id) for candidate in selected),\n        overwrite=config.overwrite,\n    )\n    _write_winning_frames(video, selected, stage_dir, config)\n    return publish_staged_video(\n        stage_dir,\n        config.output_dir,\n        video.video_id,\n        config.overwrite,\n        fast_io=config.fast_io,\n    )\n\n\ndef worker_main(\n    worker_id: int,\n    device: str,\n    videos: tuple[VideoInfo, ...],\n    config_values: dict[str, object],\n    result_queue: object,\n    cancel_event: object | None = None,\n) -> None:\n    config = RunnerConfig.from_mapping(config_values)\n    detector = load_transnet(device)\n    device_label = f"GPU {device.removeprefix(\'cuda:\')}" if device.startswith("cuda:") else "CPU"\n    print(f"worker={worker_id} device={device} videos={len(videos)}", flush=True)\n    for video in videos:\n        if config.fail_fast and cancel_event is not None and cancel_event.is_set():\n            break\n        try:\n            committed = recover_video_transaction(\n                config.output_dir,\n                video.video_id,\n                allow_legacy=config.overwrite,\n            )\n            if committed is not None and not config.overwrite:\n                result_queue.put(\n                    VideoResult(video.video_id, "skipped", len(committed), device, "")\n                )\n                continue\n            started_at = time.perf_counter()\n            raw_scenes = detector.detect_scenes(\n                str(video.path), threshold=config.scene_threshold\n            )\n            shots = normalize_scenes(raw_scenes, video)\n            paths = extract_selected_frames(video, shots, config)\n            elapsed_seconds = time.perf_counter() - started_at\n            print(\n                f"{device_label}: {video.video_id} extracted {len(paths)} keyframes "\n                f"in {elapsed_seconds:.1f}s",\n                flush=True,\n            )\n            result_queue.put(\n                VideoResult(video.video_id, "completed", len(paths), device, "")\n            )\n        except Exception as error:\n            if config.fail_fast and cancel_event is not None:\n                cancel_event.set()\n            result_queue.put(\n                VideoResult(\n                    video.video_id,\n                    "failed",\n                    0,\n                    device,\n                    f"{type(error).__name__}: {error}",\n                )\n            )\n\n\ndef _print_summary(\n    paths: Sequence[Path], devices: Sequence[str], results: Sequence[VideoResult], output_dir: Path\n) -> None:\n    counts = Counter(result.status for result in results)\n    jpeg_count = len(tuple(output_dir.glob("*.jpg"))) if output_dir.exists() else 0\n    print("devices:", ", ".join(devices))\n    print(\n        "summary:",\n        f"discovered={len(paths)}",\n        f"completed={counts[\'completed\']}",\n        f"skipped={counts[\'skipped\']}",\n        f"failed={counts[\'failed\']}",\n        f"jpeg={jpeg_count}",\n    )\n    for result in results:\n        if result.status == "failed":\n            print(f"failed video={result.video_id} device={result.device}: {result.error}")\n\n\ndef run_pipeline(config: RunnerConfig) -> tuple[VideoResult, ...]:\n    global torch\n    if torch is None:\n        torch = importlib.import_module("torch")\n    devices = plan_devices(torch.cuda.is_available(), torch.cuda.device_count(), config.max_workers)\n    paths = discover_videos(config.input_path)\n    video_ids = assign_video_ids(paths, config.input_path)\n    results: list[VideoResult] = []\n    videos: list[VideoInfo] = []\n    for path in paths:\n        video_id = video_ids[path]\n        try:\n            videos.append(probe_video(path, video_id))\n        except Exception as error:\n            results.append(\n                VideoResult(\n                    video_id,\n                    "failed",\n                    0,\n                    "coordinator",\n                    f"{type(error).__name__}: {error}",\n                )\n            )\n            if config.fail_fast:\n                _print_summary(paths, devices, results, config.output_dir)\n                return tuple(sorted(results, key=lambda item: item.video_id))\n\n    if not videos:\n        _print_summary(paths, devices, results, config.output_dir)\n        return tuple(sorted(results, key=lambda item: item.video_id))\n\n    shards = shard_videos(videos, len(devices))\n    context = multiprocessing.get_context("spawn")\n    result_queue = context.Queue()\n    cancel_event = context.Event()\n    workers = []\n    expected_results = len(videos)\n    worker_results: list[VideoResult] = []\n    intentionally_terminated: set[int] = set()\n    audited_worker_exits: set[int] = set()\n    try:\n        for worker_id, (device, shard) in enumerate(zip(devices, shards, strict=True)):\n            print(\n                f"assignment worker={worker_id} device={device} "\n                f"videos={\',\'.join(video.video_id for video in shard)}"\n            )\n            process = context.Process(\n                target=worker_main,\n                args=(\n                    worker_id,\n                    device,\n                    shard,\n                    asdict(config),\n                    result_queue,\n                    cancel_event,\n                ),\n            )\n            process.start()\n            workers.append((worker_id, device, shard, process))\n\n        while len(worker_results) < expected_results:\n            try:\n                result = result_queue.get(timeout=0.1)\n            except queue.Empty:\n                crashed = [\n                    item for item in workers\n                    if item[3].exitcode not in (None, 0)\n                    and item[0] not in intentionally_terminated\n                ]\n                if config.fail_fast and crashed:\n                    cancel_event.set()\n                    worker_id, device, shard, process = crashed[0]\n                    reported_ids = {item.video_id for item in worker_results}\n                    missing = [video for video in shard if video.video_id not in reported_ids]\n                    failed_id = missing[0].video_id if missing else f"worker-{worker_id}"\n                    worker_results.append(\n                        VideoResult(\n                            failed_id,\n                            "failed",\n                            0,\n                            device,\n                            f"worker {worker_id} exited with exit code {process.exitcode}",\n                        )\n                    )\n                    audited_worker_exits.add(worker_id)\n                    for sibling_id, _, _, sibling in workers:\n                        if sibling.is_alive():\n                            intentionally_terminated.add(sibling_id)\n                            sibling.terminate()\n                    break\n                if all(not process.is_alive() for _, _, _, process in workers):\n                    break\n                continue\n            worker_results.append(result)\n            if config.fail_fast and result.status == "failed":\n                cancel_event.set()\n    except BaseException:\n        cancel_event.set()\n        for _, _, _, process in workers:\n            if process.is_alive():\n                process.terminate()\n        raise\n    finally:\n        for _, _, _, process in workers:\n            process.join()\n\n    reported_ids = {result.video_id for result in worker_results}\n    fail_fast_failure = config.fail_fast and any(\n        result.status == "failed" for result in worker_results\n    )\n    for worker_id, device, shard, process in workers:\n        if worker_id in intentionally_terminated:\n            continue\n        missing = [video for video in shard if video.video_id not in reported_ids]\n        if process.exitcode not in (None, 0):\n            for video in missing:\n                worker_results.append(\n                    VideoResult(\n                        video.video_id,\n                        "failed",\n                        0,\n                        device,\n                        f"worker {worker_id} exited with exit code {process.exitcode}",\n                    )\n                )\n            if not missing and worker_id not in audited_worker_exits:\n                worker_results.append(\n                    VideoResult(\n                        f"worker-{worker_id}",\n                        "failed",\n                        0,\n                        device,\n                        f"worker {worker_id} exited with exit code {process.exitcode}",\n                    )\n                )\n            continue\n        if not fail_fast_failure:\n            for video in missing:\n                worker_results.append(\n                    VideoResult(\n                        video.video_id,\n                        "failed",\n                        0,\n                        device,\n                        f"worker {worker_id} returned no result",\n                    )\n                )\n\n    results.extend(worker_results)\n    ordered = tuple(sorted(results, key=lambda item: item.video_id))\n    _print_summary(paths, devices, ordered, config.output_dir)\n    return ordered\n\n\ndef main(argv: Sequence[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Extract exact TransNetV2 keyframes")\n    parser.add_argument("--input", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scene-threshold", type=float, default=0.5)\n    parser.add_argument("--coverage-seconds", type=float, default=1.0)\n    parser.add_argument("--candidate-window-seconds", type=float, default=0.25)\n    parser.add_argument("--candidate-fps", type=float, default=5.0)\n    parser.add_argument("--image-size", type=int, default=736)\n    parser.add_argument("--jpeg-quality", type=int, default=92)\n    parser.add_argument("--max-workers", type=int, default=2)\n    parser.add_argument("--overwrite", action="store_true")\n    parser.add_argument("--fail-fast", action="store_true")\n    parser.add_argument("--fast-io", action="store_true")\n    args = parser.parse_args(argv)\n    config = RunnerConfig(\n        input_path=args.input,\n        output_dir=args.output,\n        scene_threshold=args.scene_threshold,\n        coverage_seconds=args.coverage_seconds,\n        candidate_window_seconds=args.candidate_window_seconds,\n        candidate_fps=args.candidate_fps,\n        image_size=args.image_size,\n        jpeg_quality=args.jpeg_quality,\n        max_workers=args.max_workers,\n        overwrite=args.overwrite,\n        fail_fast=args.fail_fast,\n        fast_io=args.fast_io,\n    )\n    results = run_pipeline(config)\n    return 1 if any(result.status == "failed" for result in results) else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'


In [5]:
import subprocess
import sys
import time

RUNNER_PATH = Path("/kaggle/working/hcmc_keyframes_736_runner.py")


def run_keyframe_pipeline(*, clock=time.perf_counter):
    RUNNER_PATH.write_text(RUNNER_SOURCE, encoding="utf-8", newline="\n")
    command = [
        sys.executable, str(RUNNER_PATH),
        "--input", str(INPUT_PATH),
        "--output", str(OUTPUT_DIR),
        "--scene-threshold", str(SCENE_THRESHOLD),
        "--coverage-seconds", str(COVERAGE_SECONDS),
        "--candidate-window-seconds", str(CANDIDATE_WINDOW_SECONDS),
        "--candidate-fps", str(CANDIDATE_FPS),
        "--image-size", str(IMAGE_SIZE),
        "--jpeg-quality", str(JPEG_QUALITY),
        "--max-workers", str(MAX_WORKERS),
    ]
    if OVERWRITE:
        command.append("--overwrite")
    if FAIL_FAST:
        command.append("--fail-fast")
    if FAST_IO:
        command.append("--fast-io")
    started_at = clock()
    subprocess.run(command, check=True)
    elapsed_seconds = max(0.0, clock() - started_at)
    print(f"All videos completed in {format_elapsed(elapsed_seconds)}")
    return elapsed_seconds


if __name__ == "__main__":
    VIDEO_PIPELINE_SECONDS = run_keyframe_pipeline()


assignment worker=0 device=cuda:0 videos=l29-v001,l29-v002,l29-v004,l29-v006,l29-v010,l29-v011,l29-v014,l29-v015,l29-v016,l29-v018,l29-v019,l29-v023
assignment worker=1 device=cuda:1 videos=l29-v003,l29-v005,l29-v007,l29-v008,l29-v009,l29-v012,l29-v013,l29-v017,l29-v020,l29-v021,l29-v022
worker=0 device=cuda:0 videos=12
worker=1 device=cuda:1 videos=11
GPU 1: l29-v003 extracted 1407 keyframes in 402.1s
GPU 0: l29-v001 extracted 1699 keyframes in 407.3s
GPU 0: l29-v002 extracted 1581 keyframes in 395.9s
GPU 1: l29-v005 extracted 1476 keyframes in 408.6s
GPU 0: l29-v004 extracted 1559 keyframes in 395.0s
GPU 1: l29-v007 extracted 1506 keyframes in 408.3s
GPU 0: l29-v006 extracted 1586 keyframes in 403.4s
GPU 1: l29-v008 extracted 1529 keyframes in 398.1s
GPU 0: l29-v010 extracted 1565 keyframes in 390.5s
GPU 1: l29-v009 extracted 1472 keyframes in 422.3s
GPU 0: l29-v011 extracted 1419 keyframes in 388.3s
GPU 1: l29-v012 extracted 1456 keyframes in 399.4s
GPU 0: l29-v014 extracted 1483 ke

In [6]:
from __future__ import annotations

import base64
import hashlib
import json
import os
import re
import time
import zipfile
from pathlib import Path, PurePosixPath


JPEG_SUFFIXES = {".jpg", ".jpeg"}


def _archive_staging_paths(archive_path: Path) -> tuple[Path, Path]:
    archive_path = Path(archive_path)
    return (
        archive_path.with_name(f".{archive_path.name}.partial"),
        archive_path.with_name(f".{archive_path.name}.checkpoint"),
    )


def _immediate_jpeg_sources(output_dir: Path) -> list[Path]:
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return []
    if not output_dir.is_dir():
        raise NotADirectoryError(f"Keyframe output is not a directory: {output_dir}")
    return sorted(
        path
        for path in output_dir.iterdir()
        if path.is_file() and path.suffix.casefold() in JPEG_SUFFIXES
    )


def _member_name(source: Path) -> str:
    return f"keyframes/{source.name}"


def _member_infos(archive: zipfile.ZipFile) -> dict[str, zipfile.ZipInfo]:
    infos = archive.infolist()
    names = [info.filename for info in infos]
    if len(names) != len(set(names)):
        raise ValueError(f"Archive contains duplicate member names: {archive.filename}")
    for info in infos:
        member = PurePosixPath(info.filename)
        if (
            len(member.parts) != 2
            or member.parts[0] != "keyframes"
            or member.suffix.casefold() not in JPEG_SUFFIXES
            or info.is_dir()
        ):
            raise ValueError(
                f"Archive contains unexpected member {info.filename!r}: {archive.filename}"
            )
        if info.compress_type != zipfile.ZIP_STORED:
            raise ValueError(
                f"Archive member is not ZIP_STORED: {info.filename!r}"
            )
    return dict(zip(names, infos, strict=True))


def _validate_archive(archive_path: Path) -> dict[str, zipfile.ZipInfo]:
    archive_path = Path(archive_path)
    try:
        with zipfile.ZipFile(archive_path, "r", allowZip64=True) as archive:
            infos = _member_infos(archive)
            bad_member = archive.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(
                    f"CRC validation failed for {bad_member!r} in {archive_path}"
                )
            return infos
    except (OSError, ValueError, zipfile.BadZipFile) as error:
        raise type(error)(f"Invalid archive {archive_path}: {error}") from error


def _verify_archive_members(
    archive_path: Path, expected: list[tuple[str, Path]]
) -> None:
    with zipfile.ZipFile(archive_path, "r", allowZip64=True) as archive:
        infos = _member_infos(archive)
        for member_name, source in expected:
            info = infos.get(member_name)
            if info is None:
                raise ValueError(f"Missing archived member {member_name!r}")
            source_size = source.stat().st_size
            if info.file_size != source_size:
                raise ValueError(
                    f"Archived size mismatch for {member_name!r}: "
                    f"{info.file_size} != {source_size}"
                )
            archived_bytes = archive.read(info)
            source_bytes = source.read_bytes()
            if archived_bytes != source_bytes:
                raise ValueError(f"Archived content mismatch for {member_name!r}")


def _fsync_file(path: Path) -> None:
    with Path(path).open("r+b") as stream:
        os.fsync(stream.fileno())


def _fsync_directory(path: Path) -> None:
    if os.name != "posix":
        return
    flags = os.O_RDONLY | getattr(os, "O_DIRECTORY", 0)
    descriptor = os.open(os.fspath(Path(path)), flags)
    try:
        os.fsync(descriptor)
    finally:
        os.close(descriptor)


def _write_checkpoint(partial_path: Path, checkpoint_path: Path) -> None:
    valid_size = partial_path.stat().st_size
    with zipfile.ZipFile(partial_path, "r", allowZip64=True) as archive:
        _member_infos(archive)
        tail_offset = archive.start_dir
    if not 0 <= tail_offset <= valid_size:
        raise RuntimeError(f"Unsafe ZIP directory offset for {partial_path}")
    with partial_path.open("rb") as stream:
        stream.seek(tail_offset)
        tail = stream.read()
    if tail_offset + len(tail) != valid_size:
        raise RuntimeError(f"Could not checkpoint ZIP directory for {partial_path}")
    payload = {
        "version": 1,
        "valid_size": valid_size,
        "tail_offset": tail_offset,
        "tail_sha256": hashlib.sha256(tail).hexdigest(),
        "tail_base64": base64.b64encode(tail).decode("ascii"),
    }
    with checkpoint_path.open("w", encoding="utf-8", newline="\n") as stream:
        json.dump(payload, stream, sort_keys=True, separators=(",", ":"))
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    _fsync_directory(checkpoint_path.parent)


def _recover_checkpoint(partial_path: Path, checkpoint_path: Path) -> None:
    try:
        payload = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        if not isinstance(payload, dict) or payload.get("version") != 1:
            raise ValueError("unsupported checkpoint format")
        valid_size = payload["valid_size"]
        tail_offset = payload["tail_offset"]
        encoded_tail = payload["tail_base64"]
        expected_digest = payload["tail_sha256"]
        if (
            isinstance(valid_size, bool)
            or not isinstance(valid_size, int)
            or isinstance(tail_offset, bool)
            or not isinstance(tail_offset, int)
            or not isinstance(encoded_tail, str)
            or not isinstance(expected_digest, str)
        ):
            raise ValueError("checkpoint fields have invalid types")
        tail = base64.b64decode(encoded_tail, validate=True)
        if not 0 <= tail_offset <= valid_size:
            raise ValueError("checkpoint byte offsets are invalid")
        if tail_offset + len(tail) != valid_size:
            raise ValueError("checkpoint ZIP tail length is inconsistent")
        if hashlib.sha256(tail).hexdigest() != expected_digest:
            raise ValueError("checkpoint ZIP tail checksum does not match")
        if not partial_path.is_file():
            raise FileNotFoundError(f"partial archive is missing: {partial_path}")
        if partial_path.stat().st_size < tail_offset:
            raise ValueError("partial archive is shorter than the safe restore offset")
    except Exception as error:
        raise RuntimeError(
            f"Cannot recover archive checkpoint {checkpoint_path}: {error}"
        ) from error

    try:
        with partial_path.open("r+b") as stream:
            stream.seek(tail_offset)
            stream.write(tail)
            stream.truncate(valid_size)
            stream.flush()
            os.fsync(stream.fileno())
        _fsync_directory(partial_path.parent)
        _validate_archive(partial_path)
    except Exception as error:
        raise RuntimeError(
            f"Cannot safely restore {partial_path} to {valid_size} bytes: {error}"
        ) from error
    checkpoint_path.unlink()
    _fsync_directory(checkpoint_path.parent)


def _print_archive_summary(archive_path: Path, count: int) -> None:
    print("Archive path:", archive_path)
    print("Archived JPEG count:", count)
    print("Archive size bytes:", archive_path.stat().st_size)


def _transaction_manifest_entries(entries: object, video_id: str, label: str) -> list[dict[str, object]]:
    if not isinstance(entries, list):
        raise RuntimeError(f"unexpected transaction state for {video_id}: {label} is not a list")
    pattern = re.compile(rf"^{re.escape(video_id)}_s\d{{6}}_f\d{{9}}\.jpg$")
    validated: list[dict[str, object]] = []
    for entry in entries:
        if not isinstance(entry, dict) or set(entry) != {"name", "size", "sha256"}:
            raise RuntimeError(
                f"unexpected transaction state for {video_id}: invalid {label} entry"
            )
        name, size, digest = entry["name"], entry["size"], entry["sha256"]
        if (
            not isinstance(name, str)
            or pattern.fullmatch(name) is None
            or isinstance(size, bool)
            or not isinstance(size, int)
            or size < 0
            or not isinstance(digest, str)
            or re.fullmatch(r"[0-9a-f]{64}", digest) is None
        ):
            raise RuntimeError(
                f"unexpected transaction state for {video_id}: unsafe {label} entry"
            )
        validated.append({"name": name, "size": size, "sha256": digest})
    names = [entry["name"] for entry in validated]
    if names != sorted(names) or len(names) != len(set(names)):
        raise RuntimeError(
            f"unexpected transaction state for {video_id}: {label} names are not unique and sorted"
        )
    return validated


def _validate_private_directory(path: Path, allowed_names: set[str], video_id: str) -> None:
    if not path.exists():
        return
    if not path.is_dir():
        raise RuntimeError(
            f"unexpected transaction state for {video_id}: {path.name} is not a directory"
        )
    unexpected = sorted(item.name for item in path.iterdir() if not item.is_file() or item.name not in allowed_names)
    if unexpected:
        raise RuntimeError(
            f"unexpected transaction state for {video_id}: "
            f"{path.name} contains {', '.join(unexpected)}"
        )


def _read_committed_transaction_for_archive(
    transaction_dir: Path,
    archive_path: Path,
    archive_infos: dict[str, zipfile.ZipInfo],
) -> tuple[dict[str, object], Path, Path]:
    video_id = transaction_dir.name
    if re.fullmatch(r"[a-z0-9][a-z0-9-]*", video_id) is None:
        raise RuntimeError(f"unexpected transaction state directory: {transaction_dir}")
    allowed_root = {"journal.json", "staging", "backup"}
    unexpected_root = sorted(
        item.name for item in transaction_dir.iterdir() if item.name not in allowed_root
    )
    if unexpected_root:
        raise RuntimeError(
            f"unexpected transaction state for {video_id}: {', '.join(unexpected_root)}"
        )
    journal_path = transaction_dir / "journal.json"
    try:
        payload = json.loads(journal_path.read_text(encoding="utf-8"))
    except Exception as error:
        raise RuntimeError(
            f"unexpected transaction state for {video_id}: unreadable journal: {error}"
        ) from error
    if not isinstance(payload, dict) or set(payload) != {
        "version", "video_id", "phase", "old", "new_names", "manifest"
    }:
        raise RuntimeError(f"unexpected transaction state for {video_id}: invalid journal fields")
    if payload["version"] != 1 or payload["video_id"] != video_id:
        raise RuntimeError(f"unexpected transaction state for {video_id}: invalid journal identity")
    if payload["phase"] != "committed":
        raise RuntimeError(
            f"incomplete transaction for {video_id}: phase={payload['phase']!r}; "
            "recover extraction before deleting private state"
        )
    old_entries = _transaction_manifest_entries(payload["old"], video_id, "old")
    manifest = _transaction_manifest_entries(payload["manifest"], video_id, "manifest")
    new_names = payload["new_names"]
    if (
        not isinstance(new_names, list)
        or new_names != [entry["name"] for entry in manifest]
    ):
        raise RuntimeError(
            f"unexpected transaction state for {video_id}: manifest/new_names mismatch"
        )
    stage_dir = transaction_dir / "staging"
    backup_dir = transaction_dir / "backup"
    _validate_private_directory(
        stage_dir, {str(entry["name"]) for entry in manifest}, video_id
    )
    _validate_private_directory(
        backup_dir, {str(entry["name"]) for entry in old_entries}, video_id
    )
    with zipfile.ZipFile(archive_path, "r", allowZip64=True) as archive:
        for entry in manifest:
            member_name = f"keyframes/{entry['name']}"
            info = archive_infos.get(member_name)
            if info is None or info.file_size != entry["size"]:
                raise RuntimeError(
                    f"unexpected transaction state for {video_id}: "
                    f"archive manifest mismatch for {entry['name']}"
                )
            digest = hashlib.sha256()
            with archive.open(info, "r") as stream:
                for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(chunk)
            if digest.hexdigest() != entry["sha256"]:
                raise RuntimeError(
                    f"unexpected transaction state for {video_id}: "
                    f"archive checksum mismatch for {entry['name']}"
                )
    payload["old"] = old_entries
    payload["manifest"] = manifest
    return payload, stage_dir, backup_dir


def _cleanup_output_after_archive(
    output_dir: Path,
    archive_path: Path,
    archive_infos: dict[str, zipfile.ZipInfo],
) -> None:
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return
    transaction_root = output_dir / ".transactions"
    validated: list[tuple[Path, dict[str, object], Path, Path]] = []
    if transaction_root.exists():
        if not transaction_root.is_dir():
            raise RuntimeError(
                f"unexpected transaction state: {transaction_root} is not a directory"
            )
        for transaction_dir in sorted(transaction_root.iterdir(), key=lambda path: path.name):
            if not transaction_dir.is_dir():
                raise RuntimeError(
                    f"unexpected transaction state entry: {transaction_dir}"
                )
            payload, stage_dir, backup_dir = _read_committed_transaction_for_archive(
                transaction_dir, archive_path, archive_infos
            )
            validated.append((transaction_dir, payload, stage_dir, backup_dir))

    for transaction_dir, payload, stage_dir, backup_dir in validated:
        for entry in payload["manifest"]:
            staged = stage_dir / entry["name"]
            if staged.exists():
                staged.unlink()
        for entry in payload["old"]:
            saved = backup_dir / entry["name"]
            if saved.exists():
                saved.unlink()
        if stage_dir.exists():
            stage_dir.rmdir()
        if backup_dir.exists():
            backup_dir.rmdir()
        (transaction_dir / "journal.json").unlink()
        transaction_dir.rmdir()
        _fsync_directory(transaction_root)
    if transaction_root.exists():
        transaction_root.rmdir()
        _fsync_directory(output_dir)

    remaining = sorted(path.name for path in output_dir.iterdir())
    if remaining:
        raise RuntimeError(
            "Cannot remove output directory; unexpected output state remains after "
            f"archive publication: {', '.join(remaining)}"
        )
    output_dir.rmdir()
    _fsync_directory(output_dir.parent)


def archive_keyframes_low_storage(
    output_dir: Path, archive_path: Path, batch_size: int = 128
) -> tuple[Path, int]:
    output_dir = Path(output_dir)
    archive_path = Path(archive_path)
    partial_path, checkpoint_path = _archive_staging_paths(archive_path)

    source_paths = _immediate_jpeg_sources(output_dir)
    if archive_path.exists():
        if source_paths:
            raise FileExistsError(
                f"Final archive {archive_path} and source JPEG files both exist; "
                "refusing to overwrite or delete either set"
            )
        infos = _validate_archive(archive_path)
        if partial_path.exists():
            try:
                same_archive = os.path.samefile(partial_path, archive_path)
            except OSError:
                same_archive = False
            if same_archive:
                _fsync_directory(archive_path.parent)
                partial_path.unlink()
                _fsync_directory(archive_path.parent)
        _cleanup_output_after_archive(output_dir, archive_path, infos)
        _print_archive_summary(archive_path, len(infos))
        return archive_path, len(infos)

    if checkpoint_path.exists():
        _recover_checkpoint(partial_path, checkpoint_path)
    if batch_size < 1:
        raise ValueError("batch_size must be positive")

    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if partial_path.exists():
        archived_infos = _validate_archive(partial_path)
    else:
        with zipfile.ZipFile(
            partial_path, "x", compression=zipfile.ZIP_STORED, allowZip64=True
        ):
            pass
        _fsync_file(partial_path)
        _fsync_directory(partial_path.parent)
        archived_infos = {}

    duplicates = [
        (_member_name(source), source)
        for source in source_paths
        if _member_name(source) in archived_infos
    ]
    if duplicates:
        _verify_archive_members(partial_path, duplicates)
        _fsync_directory(partial_path.parent)
        for _, source in duplicates:
            source.unlink()

    source_paths = _immediate_jpeg_sources(output_dir)
    for start in range(0, len(source_paths), batch_size):
        batch = source_paths[start:start + batch_size]
        expected = [(_member_name(source), source) for source in batch]
        _write_checkpoint(partial_path, checkpoint_path)
        with zipfile.ZipFile(
            partial_path, "a", compression=zipfile.ZIP_STORED, allowZip64=True
        ) as archive:
            existing_names = set(archive.namelist())
            for member_name, source in expected:
                if member_name in existing_names:
                    raise ValueError(f"Refusing to write duplicate member {member_name!r}")
                archive.write(source, member_name, compress_type=zipfile.ZIP_STORED)
                existing_names.add(member_name)
        _fsync_file(partial_path)
        _fsync_directory(partial_path.parent)
        _verify_archive_members(partial_path, expected)
        checkpoint_path.unlink()
        _fsync_directory(checkpoint_path.parent)
        for _, source in expected:
            source.unlink()

    infos = _validate_archive(partial_path)
    if _immediate_jpeg_sources(output_dir):
        raise RuntimeError("Source JPEG files remain after archive batches completed")
    try:
        os.link(partial_path, archive_path)
    except FileExistsError as error:
        raise FileExistsError(
            f"Final archive appeared while building {partial_path}; "
            "preserving both paths without overwrite"
        ) from error
    _fsync_directory(archive_path.parent)
    partial_path.unlink()
    _fsync_directory(archive_path.parent)
    _cleanup_output_after_archive(output_dir, archive_path, infos)
    _print_archive_summary(archive_path, len(infos))
    return archive_path, len(infos)


def archive_keyframes_with_timing(
    output_dir: Path,
    archive_path: Path,
    batch_size: int,
    video_pipeline_seconds: float,
    *,
    clock=time.perf_counter,
) -> tuple[Path, int]:
    started_at = clock()
    result = archive_keyframes_low_storage(output_dir, archive_path, batch_size)
    zip_seconds = max(0.0, clock() - started_at)
    print(f"ZIP completed in {format_elapsed(zip_seconds)}")
    print(
        "End-to-end total:",
        format_elapsed(float(video_pipeline_seconds) + zip_seconds),
    )
    return result


if __name__ == "__main__":
    if "VIDEO_PIPELINE_SECONDS" not in globals():
        raise RuntimeError("Run the video extraction cell before the archive cell")
    archive_keyframes_with_timing(
        OUTPUT_DIR,
        ARCHIVE_PATH,
        ARCHIVE_BATCH_SIZE,
        VIDEO_PIPELINE_SECONDS,
    )


Archive path: /kaggle/working/keyframes.zip
Archived JPEG count: 35259
Archive size bytes: 3242334689
ZIP completed in 2m 57s
End-to-end total: 1h 20m 36s
